NAV Domain — Raw Asynchronous Log Evidence
NAV Domain — Deterministic Time Reconstruction

In [48]:
# ==============================
# SECTION 1 — INITIALIZATION
# ==============================

import sys
from pathlib import Path

# Ensure project root is visible
PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.dashboard.nav.nav_controller import NavController

MISSION_ID = "MISSION_1771663015"

# Initialize Controller (Service Layer Access Only)
ctrl = NavController(MISSION_ID)

print("Controller Initialized Successfully")

Controller Initialized Successfully


In [49]:
# ==============================
# SECTION 2 — RAW EVENT DATA
# ==============================

df_events = ctrl.data_service.get_event_audit(MISSION_ID)

print("="*70)
print("NAV DOMAIN — PRE TIME CONTROL")
print("="*70)
print(f"Mission ID     : {MISSION_ID}")
print(f"Total Packets  : {len(df_events)}")
print(f"Columns        : {list(df_events.columns)}")
print("="*70)

NAV DOMAIN — PRE TIME CONTROL
Mission ID     : MISSION_1771663015
Total Packets  : 24324
Columns        : ['mission_time', 'RelHomeAlt', 'RelOriginAlt', 'inode']


In [50]:
import duckdb
import os

db_path = os.path.expanduser("~/ardupilot-nav-domain-poc/bin/vault/warehouse_df/drone_df_views.db")
con = duckdb.connect(database=db_path, read_only=True)

# Query all columns for a mission
mission_id = "YOUR_MISSION_ID"
df_events_full = con.execute(
    "SELECT * FROM fact_nav_events WHERE mission_id = ? ORDER BY mission_time ASC",
    [mission_id]
).fetchdf()

print(df_events_full.columns.tolist())
con.close()

['mission_id', 'mission_time', 'wall_ns', 'mavpackettype', 'I', 'MagX', 'MagY', 'MagZ', 'OfsX', 'OfsY', 'OfsZ', 'MOX', 'MOY', 'MOZ', 'Health', 'S', 'inode', 'wall_ns_1', 'timestamp_sec', 'ThI', 'ABst', 'ThO', 'ThH', 'DAlt', 'Alt', 'BAlt', 'DSAlt', 'SAlt', 'TAlt', 'DCRt', 'CRt', 'DesRoll', 'Roll', 'DesPitch', 'Pitch', 'DesYaw', 'Yaw', 'AEKF', 'C', 'VN', 'VE', 'VD', 'dPD', 'PN', 'PE', 'PD', 'GX', 'GY', 'GZ', 'OH', 'Lat', 'Lng', 'Q1', 'Q2', 'Q3', 'Q4', 'Status', 'GMS', 'GWk', 'NSats', 'HDop', 'Spd', 'GCrs', 'VZ', 'U', 'RelHomeAlt', 'RelOriginAlt']


In [51]:
import pandas as pd
from src.utils.db_helpers import (
    get_silver_audit,  # NAV raw events
    get_gold_state,    # NAV aligned
    get_sys_state,     # SYS domain
    get_est_state,     # EST domain
    get_power_state,   # POWER domain
    get_comm_state     # COMM domain
)

# --- Canonical columns per domain ---
canonical_schemas = {
    "NAV": ['mission_id', 'mission_time', 'wall_ns', 'inode', 'Lat', 'Lng', 'HDop', 'NSats', 'Spd', 'VZ', 'RelHomeAlt', 'RelOriginAlt'],
    "SYS": ['mission_id', 'mission_time', 'ThI', 'ThO', 'DAlt', 'Alt', 'CRt', 'vibeX', 'vibeY', 'vibeZ'],
    "EST": ['mission_id', 'mission_time', 'pE', 'pD', 'iDX', 'iDY', 'vWN', 'vWED', 'iS'],
    "POWER": ['mission_id', 'mission_time', 'volt', 'curr', 'currTot', 'enrgTot', 'temp', 'remPct'],
    "COMM": ['mission_id', 'mission_time', 'c1', 'c2', 'c3', 'c4', 'c5', 'message']
}

# --- Map of domain -> helper ---
helper_map = {
    "NAV": get_silver_audit,
    "SYS": get_sys_state,
    "EST": get_est_state,
    "POWER": get_power_state,
    "COMM": get_comm_state
}

# --- Select a mission to test ---
mission_id = "MISSION_1771663015"  # replace with a real mission from list_missions()

# --- Audit all domains ---
for domain, helper in helper_map.items():
    df = helper(mission_id)
    expected_cols = canonical_schemas[domain]

    # Identify missing and extra columns
    missing = [c for c in expected_cols if c not in df.columns]
    extra = [c for c in df.columns if c not in expected_cols]

    # Check order
    order_ok = df.columns.tolist()[:len(expected_cols)] == expected_cols

    # Basic type check (numeric vs string)
    type_mismatch = [
        c for c in expected_cols
        if c in df.columns and not pd.api.types.is_numeric_dtype(df[c])
        and c not in ['mission_id', 'mavpackettype', 'message']
    ]

    # --- Print results ---
    print(f"\n--- {domain} ---")
    if missing:
        print(f"❌ Missing columns: {missing}")
    if extra:
        print(f"⚠ Extra columns: {extra}")
    if not order_ok:
        print("⚠ Column order mismatch")
    if type_mismatch:
        print(f"⚠ Type mismatch in: {type_mismatch}")
    if not missing and not extra and order_ok and not type_mismatch:
        print("✅ Schema OK")


--- NAV ---
⚠ Extra columns: ['mavpackettype', 'I', 'MagX', 'MagY', 'MagZ', 'OfsX', 'OfsY', 'OfsZ', 'MOX', 'MOY', 'MOZ', 'Health', 'S', 'wall_ns_1', 'timestamp_sec', 'ThI', 'ABst', 'ThO', 'ThH', 'DAlt', 'Alt', 'BAlt', 'DSAlt', 'SAlt', 'TAlt', 'DCRt', 'CRt', 'DesRoll', 'Roll', 'DesPitch', 'Pitch', 'DesYaw', 'Yaw', 'AEKF', 'C', 'VN', 'VE', 'VD', 'dPD', 'PN', 'PE', 'PD', 'GX', 'GY', 'GZ', 'OH', 'Q1', 'Q2', 'Q3', 'Q4', 'Status', 'GMS', 'GWk', 'GCrs', 'U']
⚠ Column order mismatch

--- SYS ---
❌ Missing columns: ['vibeX', 'vibeY', 'vibeZ']
⚠ Extra columns: ['wall_ns', 'mavpackettype', 'I', 'GyrX', 'GyrY', 'GyrZ', 'AccX', 'AccY', 'AccZ', 'EG', 'EA', 'T', 'GH', 'AH', 'GHz', 'AHz', 'inode', 'wall_ns_1', 'timestamp_sec', 'ABst', 'ThH', 'BAlt', 'DSAlt', 'SAlt', 'TAlt', 'DCRt', 'IMU', 'VibeX', 'VibeY', 'VibeZ', 'Clip']
⚠ Column order mismatch

--- EST ---
❌ Missing columns: ['pE', 'pD', 'iDX', 'iDY', 'vWN', 'vWED', 'iS']
⚠ Extra columns: ['wall_ns', 'mavpackettype', 'TimeUS', 'I', 'GyrX', 'GyrY',

In [52]:
import os
import duckdb

DB_PATH = os.path.expanduser("~/ardupilot-nav-domain-poc/bin/vault/warehouse_df/drone_df_views.db")

if not os.path.exists(DB_PATH):
    print(f"❌ DB not found at {DB_PATH}")
else:
    conn = duckdb.connect(DB_PATH)
    print(f"✅ Connected to {DB_PATH}")
    conn.close()

# --- Domain expected columns as per ActionMap msgids ---
expected_columns = {
    "NAV": ['mission_id', 'mission_time', 'Lat', 'Lng', 'HDop', 'NSats', 'Spd', 'VZ', 'RelHomeAlt', 'RelOriginAlt'],
    "SYS": ['mission_id', 'mission_time', 'ThI', 'ThO', 'DAlt', 'Alt', 'CRt', 'vibeX', 'vibeY', 'vibeZ'],
    "ESTIMATOR": ['mission_id', 'mission_time', 'pE', 'pD', 'iDX', 'iDY', 'vWN', 'vWED', 'iS'],
    "COM": ['mission_id', 'mission_time', 'c1', 'c2', 'c3', 'c4', 'c5', 'message']
}

# --- Map domain -> table ---
table_map = {
    "NAV": "fact_nav",
    "SYS": "fact_sys",
    "ESTIMATOR": "fact_est",
    "COM": "fact_communication"
}

conn = duckdb.connect(DB_PATH)

for domain, table in table_map.items():
    print(f"\n--- Checking {domain} ({table}) ---")
    try:
        cols = [row[0] for row in conn.execute(f"PRAGMA table_info({table})").fetchall()]
        missing = [c for c in expected_columns[domain] if c not in cols]
        extra = [c for c in cols if c not in expected_columns[domain]]

        if missing:
            print(f"❌ Missing columns: {missing}")
        if extra:
            print(f"⚠ Extra columns: {extra}")
        if not missing and not extra:
            print("✅ Table aligned with ActionMap columns")
    except Exception as e:
        print(f"❌ Could not check table {table}: {e}")

conn.close()

✅ Connected to /home/ni/ardupilot-nav-domain-poc/bin/vault/warehouse_df/drone_df_views.db

--- Checking NAV (fact_nav) ---
❌ Missing columns: ['mission_id', 'mission_time', 'Lat', 'Lng', 'HDop', 'NSats', 'Spd', 'VZ', 'RelHomeAlt', 'RelOriginAlt']
⚠ Extra columns: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66]

--- Checking SYS (fact_sys) ---
❌ Missing columns: ['mission_id', 'mission_time', 'ThI', 'ThO', 'DAlt', 'Alt', 'CRt', 'vibeX', 'vibeY', 'vibeZ']
⚠ Extra columns: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37]

--- Checking ESTIMATOR (fact_est) ---
❌ Missing columns: ['mission_id', 'mission_time', 'pE', 'pD', 'iDX', 'iDY', 'vWN', 'vWED', 'iS']
⚠ Extra columns: [0, 1, 2, 3, 4, 5,

In [53]:
import duckdb
DB_PATH = "<path_to_drone_df_views.db>"
conn = duckdb.connect(DB_PATH)

# List all tables and views
print(conn.execute("SHOW TABLES").fetchall())
print(conn.execute("SHOW VIEWS").fetchall())

[]


CatalogException: Catalog Error: Table with name VIEWS does not exist!

LINE 1: SHOW VIEWS
             ^

In [ ]:
import duckdb
import pandas as pd

# --- DB PATH ---
DB_PATH = "/home/ni/ardupilot-nav-domain-poc/bin/vault/warehouse_df/drone_df_views.db"

# --- Canonical NAV columns (MAVLink-aligned) ---
NAV_COLUMNS = [
    'mission_id', 'mission_time', 'wall_ns', 'inode',
    'Lat', 'Lng', 'HDop', 'NSats', 'Spd', 'VZ',
    'RelHomeAlt', 'RelOriginAlt'
]

# --- Connect to DB ---
conn = duckdb.connect(DB_PATH)

# --- Tables to check ---
tables = ['fact_nav', 'fact_nav_state']

for table in tables:
    print(f"\n--- Checking {table} ---")
    try:
        df = conn.execute(f"SELECT * FROM {table} LIMIT 0").fetchdf()  # only columns
        cols = df.columns.tolist()

        missing = [c for c in NAV_COLUMNS if c not in cols]
        extra = [c for c in cols if c not in NAV_COLUMNS]
        order_ok = cols[:len(NAV_COLUMNS)] == NAV_COLUMNS

        if missing:
            print(f"❌ Missing columns: {missing}")
        if extra:
            print(f"⚠ Extra columns: {extra}")
        if not order_ok:
            print(f"⚠ Column order mismatch")
        if not missing and not extra and order_ok:
            print("✅ Columns fully aligned")
    except Exception as e:
        print(f"⚠ Could not check {table}: {e}")

conn.close()


--- Checking fact_nav ---
⚠ Extra columns: ['mavpackettype', 'I', 'MagX', 'MagY', 'MagZ', 'OfsX', 'OfsY', 'OfsZ', 'MOX', 'MOY', 'MOZ', 'Health', 'S', 'wall_ns_1', 'timestamp_sec', 'ThI', 'ABst', 'ThO', 'ThH', 'DAlt', 'Alt', 'BAlt', 'DSAlt', 'SAlt', 'TAlt', 'DCRt', 'CRt', 'DesRoll', 'Roll', 'DesPitch', 'Pitch', 'DesYaw', 'Yaw', 'AEKF', 'C', 'VN', 'VE', 'VD', 'dPD', 'PN', 'PE', 'PD', 'GX', 'GY', 'GZ', 'OH', 'Q1', 'Q2', 'Q3', 'Q4', 'Status', 'GMS', 'GWk', 'GCrs', 'U']
⚠ Column order mismatch

--- Checking fact_nav_state ---
⚠ Extra columns: ['mavpackettype', 'I', 'MagX', 'MagY', 'MagZ', 'OfsX', 'OfsY', 'OfsZ', 'MOX', 'MOY', 'MOZ', 'Health', 'S', 'wall_ns_1', 'timestamp_sec', 'ThI', 'ABst', 'ThO', 'ThH', 'DAlt', 'Alt', 'BAlt', 'DSAlt', 'SAlt', 'TAlt', 'DCRt', 'CRt', 'DesRoll', 'Roll', 'DesPitch', 'Pitch', 'DesYaw', 'Yaw', 'AEKF', 'C', 'VN', 'VE', 'VD', 'dPD', 'PN', 'PE', 'PD', 'GX', 'GY', 'GZ', 'OH', 'Q1', 'Q2', 'Q3', 'Q4', 'Status', 'GMS', 'GWk', 'GCrs', 'U', 'aligned_alt', 'aligned_orig

In [ ]:
import duckdb
DB_PATH = "<path_to_drone_df_views.db>"
conn = duckdb.connect(DB_PATH)

# Check table structure
for table in ["fact_nav", "fact_nav_state"]:
    print(f"\n--- {table} ---")
    print(conn.execute(f"DESCRIBE {table}").fetchall())


--- fact_nav ---


CatalogException: Catalog Error: Table with name fact_nav does not exist!
Did you mean "duckdb_constraints"?

LINE 1: DESCRIBE fact_nav
                 ^

In [ ]:
import matplotlib.pyplot as plt

# Keep only rows where position exists
df_position_raw = df_events[
    df_events["Lat"].notna() & df_events["Lng"].notna()
]

plt.figure(figsize=(8, 8))

plt.scatter(
    df_position_raw["Lng"],
    df_position_raw["Lat"],
    s=10,
    alpha=0.5
)

plt.title("RAW NAV Events — Fragmented Trajectory")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.grid(True, linestyle=":", alpha=0.3)

plt.show()

KeyError: 'Lat'

In [ ]:
import sys
import os

# Absolute path to your project root
PROJECT_ROOT = "/home/ni/ardupilot-nav-domain-poc"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

In [ ]:
import os
import sys

# 1. FORCE THE PROJECT PATH
PROJECT_ROOT = "/home/ni/ardupilot-nav-domain-poc"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# 2. DEFINE THE AUTHORITY LOCALLY (Bypasses the broken src files)
class EmergencyActionMap:
    def __init__(self):
        # We define the "Truth" right here.
        # Add any columns you need for the NAV domain.
        self.domain_columns = {
            'NAV': [
                'TimeUS', 'Lat', 'Lng', 'Alt', 'RelHomeAlt',
                'Spd', 'GCrs', 'Roll', 'Pitch', 'Yaw', 'NSats', 'HDop'
            ]
        }

    def get_columns_for_domain(self, domain):
        return self.domain_columns.get(domain, [])

    def get_msg_types_for_domain(self, domain):
        # The messages we want the Architect to grab from the .BIN
        if domain == 'NAV':
            return ['ATT', 'POS', 'GPS', 'XKF1', 'NKF1', 'AHR2', 'CTUN', 'MAG']
        return []

# 3. INITIALIZE IT
am = EmergencyActionMap()

print("🚀 EMERGENCY AUTHORITY ACTIVE")
print(f"Targeting Columns: {am.get_columns_for_domain('NAV')}")

🚀 EMERGENCY AUTHORITY ACTIVE
Targeting Columns: ['TimeUS', 'Lat', 'Lng', 'Alt', 'RelHomeAlt', 'Spd', 'GCrs', 'Roll', 'Pitch', 'Yaw', 'NSats', 'HDop']


In [ ]:
import os
import time
import pandas as pd
from pymavlink.DFReader import DFReader_binary

# --- 1. DIRECT PATHING ---
SELECTED_BIN = "/home/ni/ardupilot-nav-domain-poc/bin/vault/df_source/clean_20260213_133342.BIN"
VAULT_B = "/home/ni/ardupilot-nav-domain-poc/bin/vault/vault_b"
os.makedirs(VAULT_B, exist_ok=True)

# --- 2. ARCHITECT DEFINITION ---
class NavDFArchitect:
    def __init__(self, bin_path, action_map):
        self.bin_path = bin_path
        self.action_map = action_map
        self.buffer = []
        self.whitelist = self.action_map.get_msg_types_for_domain('NAV')

    def record(self, msg, inode):
        d = msg.to_dict()
        cols = self.action_map.get_columns_for_domain('NAV')

        # Surgical Filter: Keep only the canonical columns
        if cols:
            d = {k: v for k, v in d.items() if k in cols}

        d['inode'] = inode
        d['wall_ns'] = time.time_ns()
        self.buffer.append(d)

        if len(self.buffer) >= 2000:
            self.flush()

    def process_flight(self):
        print(f"📖 Streaming: {os.path.basename(self.bin_path)}")
        reader = DFReader_binary(self.bin_path)
        raw_count = 0
        captured_count = 0

        while True:
            msg = reader.recv_msg()
            if msg is None: break
            raw_count += 1

            if msg.get_type() in self.whitelist:
                self.record(msg, raw_count)
                captured_count += 1

            if raw_count % 100000 == 0:
                print(f"  🔹 Scanned {raw_count} rows | Captured {captured_count} NAV...")

        self.flush()
        print(f"\n✅ EXTRACTION COMPLETE")
        print(f"Total Rows: {raw_count} | NAV Captured: {captured_count}")
        print(f"📂 Shards saved to: {VAULT_B}")

    def flush(self):
        if not self.buffer: return
        df = pd.DataFrame(self.buffer)
        shard_path = os.path.join(VAULT_B, f"nav_shard_{time.time_ns()}.parquet")
        df.to_parquet(shard_path, index=False)
        self.buffer = []

# --- 3. TRIGGER ---
arch = NavDFArchitect(SELECTED_BIN, am)
arch.process_flight()

📖 Streaming: clean_20260213_133342.BIN
  🔹 Scanned 100000 rows | Captured 5530 NAV...
  🔹 Scanned 200000 rows | Captured 10984 NAV...
  🔹 Scanned 300000 rows | Captured 16429 NAV...
  🔹 Scanned 400000 rows | Captured 21873 NAV...

✅ EXTRACTION COMPLETE
Total Rows: 444987 | NAV Captured: 24324
📂 Shards saved to: /home/ni/ardupilot-nav-domain-poc/bin/vault/vault_b


In [ ]:
📖 Streaming: clean_20260213_133342.BIN
  🔹 Scanned 100000 rows | Captured 5530 NAV...
  🔹 Scanned 200000 rows | Captured 10984 NAV...
  🔹 Scanned 300000 rows | Captured 16429 NAV...
  🔹 Scanned 400000 rows | Captured 21873 NAV...

✅ EXTRACTION COMPLETE
Total Rows: 444987 | NAV Captured: 24324
📂 Shards saved to: /home/ni/ardupilot-nav-domain-poc/bin/vault/vault_b

📂 Found Log: clean_20260213_133342.BIN
❌ ERROR: 'am' (ActionMap) is not defined. Run your ActionMap cell first.


In [ ]:
import os
import sys
import time
import pandas as pd
from pymavlink.DFReader import DFReader_binary

# 1. FORCE THE PATH
PROJECT_ROOT = "/home/ni/ardupilot-nav-domain-poc"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# 2. DEFINE THE ARCHITECT LOCALLY (Bypasses Import Errors)
class NavDFArchitect:
    def __init__(self, bin_path, action_map, limit=5000):
        self.bin_path = bin_path
        self.action_map = action_map
        self.limit = 5000
        self.buffer = []
        self.vault_b = os.path.join(PROJECT_ROOT, "bin/vault/vault_b")
        os.makedirs(self.vault_b, exist_ok=True)

        # Hardcoded for speed now, but derived from ActionMap logic
        self.whitelist = ['ATT', 'POS', 'GPS', 'XKF1', 'NKF1', 'AHR2', 'CTUN', 'MAG']

    def record(self, msg, inode):
        d = msg.to_dict()
        msg_type = msg.get_type()

        # Call the Materializer via ActionMap for schema enforcement
        canonical_cols = self.action_map.get_columns_for_domain('NAV')
        if canonical_cols:
            # Drop the ghost columns (mavpackettype, I, etc.) here
            d = {k: v for k, v in d.items() if k in canonical_cols}

        d['inode'] = inode
        d['wall_ns'] = time.time_ns()
        if 'TimeUS' in d:
            d['timestamp_sec'] = d['TimeUS'] / 1e6

        self.buffer.append(d)
        if len(self.buffer) >= self.limit:
            self.flush()

    def process_flight(self):
        if not os.path.exists(self.bin_path):
            print(f"❌ BIN NOT FOUND: {self.bin_path}")
            return False

        reader = DFReader_binary(self.bin_path)
        raw_count = 0
        captured_count = 0
        print(f"📖 Streaming {os.path.basename(self.bin_path)}...")

        while True:
            msg = reader.recv_msg()
            if msg is None: break
            raw_count += 1
            if msg.get_type() in self.whitelist:
                self.record(msg, raw_count)
                captured_count += 1

        self.flush()
        print(f"✅ Finished. Sharded {captured_count} NAV messages to Vault B.")
        return True

    def flush(self):
        if not self.buffer: return
        df = pd.DataFrame(self.buffer)
        shard_path = os.path.join(self.vault_b, f"nav_shard_{time.time_ns()}.parquet")
        df.to_parquet(shard_path, index=False)
        self.buffer = []

# 3. EXECUTION
# Ensure your ActionMap and Materializers are already initialized in your notebook
# am = ActionMap(switch, materializers)
# arch = NavDFArchitect("path/to/your/log.bin", am)
# arch.process_flight()

In [61]:
# 1. Update the 'am' authority with COM schema and message types
am.domain_columns['COM'] = [
    'TimeUS', 'RSSI', 'RemRSSI', 'TxBuf', 'Noise', 'RemNoise'
]

# 2. Map the MAVLink/DataFlash message types to the COM domain
# ArduPilot uses 'RADIO' or 'RAD' for link telemetry
com_whitelist = ['RADIO', 'RAD']

print(f"📡 COM Authority Active. Targeting: {am.domain_columns['COM']}")

📡 COM Authority Active. Targeting: ['TimeUS', 'RSSI', 'RemRSSI', 'TxBuf', 'Noise', 'RemNoise']


In [ ]:
import os
import time
import pandas as pd
from pymavlink.DFReader import DFReader_binary

# --- CONFIG ---
SELECTED_BIN = "/home/ni/ardupilot-nav-domain-poc/bin/vault/df_source/clean_20260213_133342.BIN"
VAULT_B = "/home/ni/ardupilot-nav-domain-poc/bin/vault/vault_b"
WAREHOUSE_DF = "/home/ni/ardupilot-nav-domain-poc/bin/vault/warehouse_df"

class ComDFArchitect:
    def __init__(self, bin_path, action_map):
        self.bin_path = bin_path
        self.am = action_map
        self.buffer = []
        # RADIO/RAD are the standard COM messages in DataFlash
        self.whitelist = ['RADIO', 'RAD']

    def flush(self):
        if not self.buffer: return
        df = pd.DataFrame(self.buffer)
        # Unique naming to avoid mixing with NAV shards
        shard_path = os.path.join(VAULT_B, f"com_shard_{time.time_ns()}.parquet")
        df.to_parquet(shard_path, index=False)
        self.buffer = []

    def process(self):
        print(f"🚀 Extracting COM from: {os.path.basename(self.bin_path)}")
        reader = DFReader_binary(self.bin_path)
        raw_count = 0
        captured_count = 0

        # Pull the 'Truth' columns from the authority
        cols = self.am.domain_columns['COM']

        while True:
            msg = reader.recv_msg()
            if msg is None: break
            raw_count += 1

            if msg.get_type() in self.whitelist:
                d = msg.to_dict()
                # ENFORCEMENT: Only keep the 6 columns defined in 'am'
                filtered_d = {k: v for k, v in d.items() if k in cols}

                # Metadata
                filtered_d['inode'] = raw_count
                filtered_d['wall_ns'] = time.time_ns()

                self.buffer.append(filtered_d)
                captured_count += 1

                if len(self.buffer) >= 1000: self.flush()

        self.flush()
        print(f"✅ COM Capture Complete: {captured_count} rows.")

# --- RUN ARCHITECT ---
com_arch = ComDFArchitect(SELECTED_BIN, am)
com_arch.process()

In [ ]:
import duckdb
import os

# 1. Point to your warehouse
WAREHOUSE = "/home/ni/ardupilot-nav-domain-poc/bin/vault/warehouse_df"

# 2. Find the latest master file we just created
master_files = [os.path.join(WAREHOUSE, f) for f in os.listdir(WAREHOUSE) if f.endswith('.parquet')]
if not master_files:
    print("❌ No Master Parquets found in the warehouse!")
else:
    # Get the most recently created file
    latest_master = max(master_files, key=os.path.getctime)
    print(f"🧐 Inspecting Gold Master: {os.path.basename(latest_master)}")

    # 3. Query the schema and data
    con = duckdb.connect(':memory:')
    df_check = con.execute(f"SELECT * FROM read_parquet('{latest_master}') LIMIT 5").df()

    print("\n📊 FINAL WAREHOUSE COLUMNS:")
    print(df_check.columns.tolist())

    print("\n💎 DATA PREVIEW:")
    display(df_check)

🧐 Inspecting Gold Master: com_df_master.parquet

📊 FINAL WAREHOUSE COLUMNS:
['mission_id', 'mission_time', 'wall_ns', 'mavpackettype', 'ID', 'Seq', 'Message', 'inode', 'wall_ns_1', 'timestamp_sec', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'Id_1']

💎 DATA PREVIEW:


,mission_id,mission_time,wall_ns,mavpackettype,ID,Seq,Message,inode,wall_ns_1,timestamp_sec,...,C6,C7,C8,C9,C10,C11,C12,C13,C14,Id_1
0,MISSION_1771663015,2.444855,2444855000,MSG,10.0,0.0,ArduCopter V4.7.0-dev (d56f4839),1785,1771662094596044402,2.444855,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,MISSION_1771663015,2.444855,2444855000,MSG,11.0,0.0,6e856b4179f0474a9acf9b4e7eb30867,1787,1771662094596077697,2.444855,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,MISSION_1771663015,2.444855,2444855000,MSG,12.0,0.0,Param space used: 327/4096,1788,1771662094596092941,2.444855,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,MISSION_1771663015,2.444855,2444855000,MSG,13.0,0.0,RC Protocol: UDP,1789,1771662094596105677,2.444855,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,MISSION_1771663015,2.444855,2444855000,MSG,14.0,0.0,New mission,1790,1771662094596117912,2.444855,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
import os
import time
import pandas as pd
from pymavlink.DFReader import DFReader_binary

# --- CONFIG ---
SELECTED_BIN = "/home/ni/ardupilot-nav-domain-poc/bin/vault/df_source/clean_20260213_133342.BIN"
VAULT_B = "/home/ni/ardupilot-nav-domain-poc/bin/vault/vault_b"
WAREHOUSE_DF = "/home/ni/ardupilot-nav-domain-poc/bin/vault/warehouse_df"

class ComDFArchitect:
    def __init__(self, bin_path, action_map):
        self.bin_path = bin_path
        self.am = action_map
        self.buffer = []
        # RADIO/RAD are the standard COM messages in DataFlash
        self.whitelist = ['RADIO', 'RAD']

    def flush(self):
        if not self.buffer: return
        df = pd.DataFrame(self.buffer)
        # Unique naming to avoid mixing with NAV shards
        shard_path = os.path.join(VAULT_B, f"com_shard_{time.time_ns()}.parquet")
        df.to_parquet(shard_path, index=False)
        self.buffer = []

    def process(self):
        print(f"🚀 Extracting COM from: {os.path.basename(self.bin_path)}")
        reader = DFReader_binary(self.bin_path)
        raw_count = 0
        captured_count = 0

        # Pull the 'Truth' columns from the authority
        cols = self.am.domain_columns['COM']

        while True:
            msg = reader.recv_msg()
            if msg is None: break
            raw_count += 1

            if msg.get_type() in self.whitelist:
                d = msg.to_dict()
                # ENFORCEMENT: Only keep the 6 columns defined in 'am'
                filtered_d = {k: v for k, v in d.items() if k in cols}

                # Metadata
                filtered_d['inode'] = raw_count
                filtered_d['wall_ns'] = time.time_ns()

                self.buffer.append(filtered_d)
                captured_count += 1

                if len(self.buffer) >= 1000: self.flush()

        self.flush()
        print(f"✅ COM Capture Complete: {captured_count} rows.")

# --- RUN ARCHITECT ---
com_arch = ComDFArchitect(SELECTED_BIN, am)
com_arch.process()

🚀 Extracting COM from: clean_20260213_133342.BIN
✅ COM Capture Complete: 0 rows.


In [63]:
from pymavlink.DFReader import DFReader_binary

reader = DFReader_binary(SELECTED_BIN)
found_types = set()
for i in range(100000):
    msg = reader.recv_msg()
    if msg is None: break
    found_types.add(msg.get_type())

print("🔍 MESSAGE TYPES FOUND IN THIS LOG:")
print(sorted(list(found_types)))

🔍 MESSAGE TYPES FOUND IN THIS LOG:
['AHR2', 'ANG', 'ATT', 'AUXF', 'BARO', 'BAT', 'CMD', 'CTUN', 'DCM', 'DSF', 'DU32', 'ERR', 'ESC', 'ESCX', 'EV', 'FMT', 'FMTU', 'GPA', 'GPS', 'IMU', 'MAG', 'MAV', 'MAVC', 'MODE', 'MOTB', 'MSG', 'MULT', 'ORGN', 'PARM', 'PIDA', 'PIDP', 'PIDR', 'PIDY', 'PM', 'POS', 'RATE', 'RCI2', 'RCIN', 'RCO2', 'RCOU', 'SIM', 'SIM2', 'SRTL', 'SURF', 'TERR', 'UART', 'UNIT', 'VER', 'VIBE', 'XKF1', 'XKF2', 'XKF3', 'XKF4', 'XKF5', 'XKFS', 'XKQ', 'XKT', 'XKTV', 'XKV1', 'XKV2']


In [64]:
# 1. Define SYS schema (System Health)
am.domain_columns['SYS'] = [
    'TimeUS', 'NLon', 'FreeMem', 'Load', 'MaxT'
]

# 2. Define whitelist for SYS based on your 'PM' message
sys_whitelist = ['PM']

print(f"🖥️ SYS Authority Active. Targeting: {am.domain_columns['SYS']}")

🖥️ SYS Authority Active. Targeting: ['TimeUS', 'NLon', 'FreeMem', 'Load', 'MaxT']


In [ ]:
import os
import time
import pandas as pd
from pymavlink.DFReader import DFReader_binary

# --- CONFIG ---
SELECTED_BIN = "/home/ni/ardupilot-nav-domain-poc/bin/vault/df_source/clean_20260213_133342.BIN"
VAULT_B = "/home/ni/ardupilot-nav-domain-poc/bin/vault/vault_b"

class SysDFArchitect:
    def __init__(self, bin_path, action_map):
        self.bin_path = bin_path
        self.am = action_map
        self.buffer = []
        self.whitelist = ['PM'] # Confirmed present in your discovery scan

    def flush(self):
        if not self.buffer: return
        df = pd.DataFrame(self.buffer)
        shard_path = os.path.join(VAULT_B, f"sys_shard_{time.time_ns()}.parquet")
        df.to_parquet(shard_path, index=False)
        self.buffer = []

    def process(self):
        print(f"🚀 Extracting SYS (PM) from: {os.path.basename(self.bin_path)}")
        reader = DFReader_binary(self.bin_path)
        raw_count = 0
        captured_count = 0

        # Get the 'Truth' columns for SYS
        cols = self.am.domain_columns['SYS']

        while True:
            msg = reader.recv_msg()
            if msg is None: break
            raw_count += 1

            if msg.get_type() in self.whitelist:
                d = msg.to_dict()
                # Surgical filter: Keep only TimeUS, NLon, FreeMem, Load, MaxT
                filtered_d = {k: v for k, v in d.items() if k in cols}

                # Add Metadata
                filtered_d['inode'] = raw_count
                filtered_d['wall_ns'] = time.time_ns()

                self.buffer.append(filtered_d)
                captured_count += 1

                if len(self.buffer) >= 1000: self.flush()

        self.flush()
        print(f"✅ SYS Capture Complete: {captured_count} rows.")

# --- RUN ---
sys_arch = SysDFArchitect(SELECTED_BIN, am)
sys_arch.process()

🚀 Extracting SYS (PM) from: clean_20260213_133342.BIN
✅ SYS Capture Complete: 25 rows.


In [67]:
import os
import duckdb

# Paths
VAULT_B = "/home/ni/ardupilot-nav-domain-poc/bin/vault/vault_b"
MASTER_SYS = "/home/ni/ardupilot-nav-domain-poc/bin/vault/warehouse_df/sys_df_master.parquet"

# 1. Consolidate to Gold Master
shards = [os.path.join(VAULT_B, f) for f in os.listdir(VAULT_B) if 'sys_shard' in f]
if shards:
    con = duckdb.connect(':memory:')
    con.execute(f"COPY (SELECT * FROM read_parquet({shards}) ORDER BY inode ASC) TO '{MASTER_SYS}' (FORMAT 'PARQUET')")
    print(f"💎 Gold SYS Master Saved: {MASTER_SYS}")

# 2. Final Wipe of Vault B (Cleanup)
all_shards = [os.path.join(VAULT_B, f) for f in os.listdir(VAULT_B) if f.endswith('.parquet')]
for s in all_shards:
    os.remove(s)

print(f"🧹 Vault B Cleared ({len(all_shards)} shards removed).")
print("✅ MISSION COMPLETE: Warehouse is clean and structured.")

💎 Gold SYS Master Saved: /home/ni/ardupilot-nav-domain-poc/bin/vault/warehouse_df/sys_df_master.parquet
🧹 Vault B Cleared (15 shards removed).
✅ MISSION COMPLETE: Warehouse is clean and structured.
